In [2]:
from pathlib import Path
from collections import Counter

files = list(RAW_DATA_DIR.rglob("*"))

print("Total files:", len(files))

extensions = Counter(
    file.suffix.lower()
    for file in files
    if file.is_file()
)

extensions

Total files: 1843


Counter({'.txt': 922, '.wav': 920, '.sh': 1})

In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw" / "ICBHI"
INTERIM_DATA_DIR = DATA_DIR / "interim"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

print("Project:", PROJECT_ROOT)
print("Raw data:", RAW_DATA_DIR)

Project: /Users/vs/Sem 5/Respiratory-Disease-Classification
Raw data: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/ICBHI


In [3]:
wav_files = sorted(RAW_DATA_DIR.rglob("*.wav"))

print("WAV files:", len(wav_files))

for file in wav_files[:10]:
    print(file.name)


WAV files: 920
101_1b1_Al_sc_Meditron.wav
101_1b1_Pr_sc_Meditron.wav
102_1b1_Ar_sc_Meditron.wav
103_2b2_Ar_mc_LittC2SE.wav
104_1b1_Al_sc_Litt3200.wav
104_1b1_Ar_sc_Litt3200.wav
104_1b1_Ll_sc_Litt3200.wav
104_1b1_Lr_sc_Litt3200.wav
104_1b1_Pl_sc_Litt3200.wav
104_1b1_Pr_sc_Litt3200.wav


In [4]:
annotation_files = sorted(RAW_DATA_DIR.rglob("*.txt"))

print("TXT files:", len(annotation_files))

for file in annotation_files[:10]:
    print(file.name)

TXT files: 922
101_1b1_Al_sc_Meditron.txt
101_1b1_Pr_sc_Meditron.txt
102_1b1_Ar_sc_Meditron.txt
103_2b2_Ar_mc_LittC2SE.txt
104_1b1_Al_sc_Litt3200.txt
104_1b1_Ar_sc_Litt3200.txt
104_1b1_Ll_sc_Litt3200.txt
104_1b1_Lr_sc_Litt3200.txt
104_1b1_Pl_sc_Litt3200.txt
104_1b1_Pr_sc_Litt3200.txt


In [14]:
parse_icbhi_filename(
    "101_1b1_Al_sc_Meditron.wav"
)

{'patient_id': 101,
 'recording_id': '1b1',
 'chest_location': 'Al',
 'acquisition_mode': 'sc',
 'equipment': 'Meditron'}

In [18]:
test_file = "101_1b1_Al_sc_Meditron.wav"

parsed = parse_icbhi_filename(test_file)

print(parsed)

{'patient_id': 101, 'recording_id': '1b1', 'chest_location': 'Al', 'acquisition_mode': 'sc', 'equipment': 'Meditron'}


In [22]:
from pathlib import Path
import sys

# Project root directory
PROJECT_ROOT = Path.cwd().parent

# Add project root to Python's import path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: /Users/vs/Sem 5/Respiratory-Disease-Classification


In [23]:
from src.data.validator import validate_parsed_record

In [24]:
print(validate_parsed_record)

<function validate_parsed_record at 0x10f5f5590>


In [25]:
test_record = {
    "patient_id": 101,
    "recording_id": "1b1",
    "chest_location": "Al",
    "acquisition_mode": "sc",
    "equipment": "Meditron"
}

errors = validate_parsed_record(test_record)

print(errors)

[]


In [26]:
bad_record = {
    "patient_id": 101,
    "recording_id": "1b1",
    "chest_location": "XYZ",
    "acquisition_mode": "abc",
    "equipment": "Unknown"
}

errors = validate_parsed_record(bad_record)

print(errors)

['invalid_location', 'invalid_mode', 'invalid_equipment']


In [30]:
wav_files = sorted(RAW_DATA_DIR.rglob("*.wav"))

In [31]:
validation_results = []

for wav_file in wav_files:

    try:
        record = parse_icbhi_filename(wav_file.name)

        errors = validate_parsed_record(record)

        validation_results.append({
            "file": wav_file.name,
            "valid": len(errors) == 0,
            "errors": errors
        })

    except Exception as e:

        validation_results.append({
            "file": wav_file.name,
            "valid": False,
            "errors": [str(e)]
        })

In [32]:
import pandas as pd

validation_df = pd.DataFrame(validation_results)

validation_df.head()

,file,valid,errors
0,101_1b1_Al_sc_Meditron.wav,True,[]
1,101_1b1_Pr_sc_Meditron.wav,True,[]
2,102_1b1_Ar_sc_Meditron.wav,True,[]
3,103_2b2_Ar_mc_LittC2SE.wav,True,[]
4,104_1b1_Al_sc_Litt3200.wav,True,[]


In [33]:
print("Total WAV files:", len(validation_df))
print("Valid files:", validation_df["valid"].sum())
print("Invalid files:", (~validation_df["valid"]).sum())

Total WAV files: 920
Valid files: 920
Invalid files: 0


In [34]:
validation_df["valid"].value_counts()

valid
True    920
Name: count, dtype: int64

In [35]:
validation_df[validation_df["valid"] == False]

,file,valid,errors


In [36]:
records = []

for wav_file in wav_files:

    record = parse_icbhi_filename(wav_file.name)

    record["wav_path"] = str(wav_file)

    records.append(record)

recordings_df = pd.DataFrame(records)

In [37]:
recordings_df.head()

,patient_id,recording_id,chest_location,acquisition_mode,equipment,wav_path
0,101,1b1,Al,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
1,101,1b1,Pr,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
2,102,1b1,Ar,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
3,103,2b2,Ar,mc,LittC2SE,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
4,104,1b1,Al,sc,Litt3200,/Users/vs/Sem 5/Respiratory-Disease-Classifica...


In [38]:
print("Total recordings:", len(recordings_df))
print("Unique patients:", recordings_df["patient_id"].nunique())

Total recordings: 920
Unique patients: 126


In [39]:
recordings_df["chest_location"].value_counts()

chest_location
Ar    168
Al    162
Pl    139
Pr    132
Tc    130
Lr    112
Ll     77
Name: count, dtype: int64

In [40]:
recordings_df["acquisition_mode"].value_counts()

acquisition_mode
mc    732
sc    188
Name: count, dtype: int64

In [41]:
recordings_df["equipment"].value_counts()

equipment
AKGC417L    646
Meditron    127
LittC2SE     87
Litt3200     60
Name: count, dtype: int64

In [42]:
recordings_df["patient_id"].nunique()

126

In [43]:
recordings_df.to_csv(
    INTERIM_DATA_DIR / "recordings.csv",
    index=False
)

In [44]:
recordings_df.isnull().sum()

patient_id          0
recording_id        0
chest_location      0
acquisition_mode    0
equipment           0
wav_path            0
dtype: int64

In [45]:
print(
    "Duplicate WAV paths:",
    recordings_df["wav_path"].duplicated().sum()
)

Duplicate WAV paths: 0


In [46]:
print(
    "Duplicate recording IDs:",
    recordings_df["recording_id"].duplicated().sum()
)

Duplicate recording IDs: 886


In [47]:
recordings_per_patient = (
    recordings_df
    .groupby("patient_id")
    .size()
    .sort_values()
)

recordings_per_patient.head()

patient_id
164    1
208    1
125    1
126    1
127    1
dtype: int64

In [48]:
recordings_per_patient.describe()

count    126.000000
mean       7.301587
std        9.519050
min        1.000000
25%        1.000000
50%        3.000000
75%        8.000000
max       66.000000
dtype: float64

In [49]:
test_recordings_df = pd.read_csv(
    INTERIM_DATA_DIR / "recordings.csv"
)

print(test_recordings_df.shape)

(920, 6)


In [50]:
test_recordings_df.head()

,patient_id,recording_id,chest_location,acquisition_mode,equipment,wav_path
0,101,1b1,Al,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
1,101,1b1,Pr,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
2,102,1b1,Ar,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
3,103,2b2,Ar,mc,LittC2SE,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
4,104,1b1,Al,sc,Litt3200,/Users/vs/Sem 5/Respiratory-Disease-Classifica...


In [51]:
print(
    "Duplicate WAV filenames:",
    recordings_df["wav_path"].map(lambda x: Path(x).name).duplicated().sum()
)

Duplicate WAV filenames: 0


In [52]:
annotation_files = sorted(RAW_DATA_DIR.rglob("*.txt"))

print("Total TXT files:", len(annotation_files))

for file in annotation_files[:20]:
    print(file)

Total TXT files: 922
/Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/ICBHI/101_1b1_Al_sc_Meditron.txt
/Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/ICBHI/101_1b1_Pr_sc_Meditron.txt
/Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/ICBHI/102_1b1_Ar_sc_Meditron.txt
/Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/ICBHI/103_2b2_Ar_mc_LittC2SE.txt
/Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/ICBHI/104_1b1_Al_sc_Litt3200.txt
/Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/ICBHI/104_1b1_Ar_sc_Litt3200.txt
/Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/ICBHI/104_1b1_Ll_sc_Litt3200.txt
/Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/ICBHI/104_1b1_Lr_sc_Litt3200.txt
/Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/ICBHI/104_1b1_Pl_sc_Litt3200.txt
/Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/ICBHI/104_1b1_Pr_sc_Litt3200.txt
/Users/vs/Sem 5/Respiratory-Disease-Classificatio

In [53]:
wav_stems = {
    file.stem
    for file in wav_files
}

txt_stems = {
    file.stem
    for file in annotation_files
}

extra_txt_files = sorted(txt_stems - wav_stems)

print("TXT files without matching WAV:", len(extra_txt_files))

for stem in extra_txt_files:
    print(stem)

TXT files without matching WAV: 2
filename_differences
filename_format


In [54]:
for stem in extra_txt_files:
    print(stem)

filename_differences
filename_format


In [55]:
annotation_files = sorted(
    file for file in RAW_DATA_DIR.rglob("*.txt")
    if file.stem in wav_stems
)

print("Recording annotation files:", len(annotation_files))

Recording annotation files: 920


In [56]:
annotation_stems = {
    file.stem
    for file in annotation_files
}

missing_annotations = sorted(wav_stems - annotation_stems)

print("WAV files without annotations:", len(missing_annotations))

for stem in missing_annotations:
    print(stem)

WAV files without annotations: 0


In [57]:
sample_annotation = annotation_files[0]

print("File:", sample_annotation)
print()
print(sample_annotation.read_text())

File: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/ICBHI/101_1b1_Al_sc_Meditron.txt

0.036	0.579	0	0
0.579	2.45	0	0
2.45	3.893	0	0
3.893	5.793	0	0
5.793	7.521	0	0
7.521	9.279	0	0
9.279	11.15	0	0
11.15	13.036	0	0
13.036	14.721	0	0
14.721	16.707	0	0
16.707	18.507	0	0
18.507	19.964	0	0



In [58]:
sample_annotation = annotation_files[0]

print("File:", sample_annotation)

with open(sample_annotation, "r") as f:
    for i, line in enumerate(f):
        print(repr(line))
        if i >= 9:
            break

File: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/ICBHI/101_1b1_Al_sc_Meditron.txt
'0.036\t0.579\t0\t0\n'
'0.579\t2.45\t0\t0\n'
'2.45\t3.893\t0\t0\n'
'3.893\t5.793\t0\t0\n'
'5.793\t7.521\t0\t0\n'
'7.521\t9.279\t0\t0\n'
'9.279\t11.15\t0\t0\n'
'11.15\t13.036\t0\t0\n'
'13.036\t14.721\t0\t0\n'
'14.721\t16.707\t0\t0\n'


In [64]:
import importlib
import src.data.parser as parser

In [65]:
print(hasattr(parser, "parse_icbhi_annotation"))

True


In [66]:
importlib.reload(parser)

<module 'src.data.parser' from '/Users/vs/Sem 5/Respiratory-Disease-Classification/src/data/parser.py'>

In [67]:
print(hasattr(parser, "parse_icbhi_annotation"))

True


In [68]:
from src.data.parser import (
    parse_icbhi_filename,
    parse_icbhi_annotation
)

In [69]:
sample_annotation = annotation_files[0]

cycles = parse_icbhi_annotation(sample_annotation)

print("Number of cycles:", len(cycles))
print(cycles[:5])

Number of cycles: 12
[{'start_time': 0.036, 'end_time': 0.579, 'crackles': 0, 'wheezes': 0}, {'start_time': 0.579, 'end_time': 2.45, 'crackles': 0, 'wheezes': 0}, {'start_time': 2.45, 'end_time': 3.893, 'crackles': 0, 'wheezes': 0}, {'start_time': 3.893, 'end_time': 5.793, 'crackles': 0, 'wheezes': 0}, {'start_time': 5.793, 'end_time': 7.521, 'crackles': 0, 'wheezes': 0}]


In [70]:
annotation_validation_results = []

for annotation_file in annotation_files:

    try:
        cycles = parse_icbhi_annotation(annotation_file)

        annotation_validation_results.append({
            "file": annotation_file.name,
            "valid": True,
            "num_cycles": len(cycles),
            "error": None
        })

    except Exception as e:

        annotation_validation_results.append({
            "file": annotation_file.name,
            "valid": False,
            "num_cycles": 0,
            "error": str(e)
        })

In [71]:
annotation_validation_df = pd.DataFrame(
    annotation_validation_results
)

In [72]:
print("Total annotation files:", len(annotation_validation_df))
print(
    "Valid:",
    annotation_validation_df["valid"].sum()
)
print(
    "Invalid:",
    (~annotation_validation_df["valid"]).sum()
)

Total annotation files: 920
Valid: 920
Invalid: 0


In [73]:
annotation_validation_df["num_cycles"].describe()

count    920.000000
mean       7.497826
std        3.335762
min        2.000000
25%        5.000000
50%        7.000000
75%        9.000000
max       33.000000
Name: num_cycles, dtype: float64

In [74]:
annotation_validation_df[
    annotation_validation_df["valid"] == False
]

,file,valid,num_cycles,error


In [78]:
import importlib
import src.data.validator as validator

importlib.reload(validator)

<module 'src.data.validator' from '/Users/vs/Sem 5/Respiratory-Disease-Classification/src/data/validator.py'>

In [79]:
from src.data.validator import validate_annotation_cycle

In [80]:
print(hasattr(validator, "validate_annotation_cycle"))

True


In [81]:
from src.data.validator import validate_annotation_cycle

In [82]:
test_cycle = {
    "start_time": 0.036,
    "end_time": 0.579,
    "crackles": 0,
    "wheezes": 0
}

print(validate_annotation_cycle(test_cycle))

[]


In [83]:
bad_cycle = {
    "start_time": 2.0,
    "end_time": 1.0,
    "crackles": 2,
    "wheezes": -1
}

print(validate_annotation_cycle(bad_cycle))

['invalid_time_range', 'invalid_crackles_value', 'invalid_wheezes_value']


In [85]:
cycle_validation_results = []

for annotation_file in annotation_files:

    cycles = parse_icbhi_annotation(annotation_file)

    for cycle_number, cycle in enumerate(cycles, start=1):

        errors = validate_annotation_cycle(cycle)

        cycle_validation_results.append({
            "file": annotation_file.name,
            "cycle_number": cycle_number,
            "start_time": cycle["start_time"],
            "end_time": cycle["end_time"],
            "crackles": cycle["crackles"],
            "wheezes": cycle["wheezes"],
            "valid": len(errors) == 0,
            "errors": errors
        })

In [86]:
cycle_validation_df = pd.DataFrame(
    cycle_validation_results
)

In [87]:
print("Total respiratory cycles:", len(cycle_validation_df))

Total respiratory cycles: 6898


In [88]:
print(
    "Valid cycles:",
    cycle_validation_df["valid"].sum()
)

print(
    "Invalid cycles:",
    (~cycle_validation_df["valid"]).sum()
)

Valid cycles: 6898
Invalid cycles: 0


In [89]:
cycle_validation_df[
    cycle_validation_df["valid"] == False
]

,file,cycle_number,start_time,end_time,crackles,wheezes,valid,errors


In [90]:
cycle_validation_df[
    ["crackles", "wheezes"]
].value_counts()

crackles  wheezes
0         0          3642
1         0          1864
0         1           886
1         1           506
Name: count, dtype: int64

In [91]:
cycle_validation_df.to_csv(
    INTERIM_DATA_DIR / "cycles_validation.csv",
    index=False
)

In [92]:
cycle_validation_df.to_csv(
    INTERIM_DATA_DIR / "cycles_validation.csv",
    index=False
)

print("Saved:", INTERIM_DATA_DIR / "cycles_validation.csv")
print("Rows:", len(cycle_validation_df))

Saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/interim/cycles_validation.csv
Rows: 6898


In [93]:
import soundfile as sf

In [94]:
sample_wav = wav_files[0]

info = sf.info(sample_wav)

print("File:", sample_wav.name)
print("Sample rate:", info.samplerate)
print("Frames:", info.frames)
print("Duration:", info.duration)

File: 101_1b1_Al_sc_Meditron.wav
Sample rate: 44100
Frames: 882000
Duration: 20.0


In [95]:
audio_info = []

for wav_file in wav_files:

    info = sf.info(wav_file)

    audio_info.append({
        "file": wav_file.name,
        "sample_rate": info.samplerate,
        "frames": info.frames,
        "duration": info.duration
    })

audio_info_df = pd.DataFrame(audio_info)

In [96]:
audio_info_df.head()

,file,sample_rate,frames,duration
0,101_1b1_Al_sc_Meditron.wav,44100,882000,20.000
1,101_1b1_Pr_sc_Meditron.wav,44100,882000,20.000
2,102_1b1_Ar_sc_Meditron.wav,44100,882000,20.000
3,103_2b2_Ar_mc_LittC2SE.wav,44100,882000,20.000
4,104_1b1_Al_sc_Litt3200.wav,4000,63424,15.856


In [97]:
print("Total audio files:", len(audio_info_df))

Total audio files: 920


In [98]:
audio_info_df["sample_rate"].value_counts()

sample_rate
44100    824
4000      90
10000      6
Name: count, dtype: int64

In [99]:
audio_info_df["duration"].describe()

count    920.000000
mean      21.492424
std        8.307421
min        7.856000
25%       20.000000
50%       20.000000
75%       20.000000
max       86.200000
Name: duration, dtype: float64

In [100]:
audio_info_df.to_csv(
    INTERIM_DATA_DIR / "audio_info.csv",
    index=False
)

In [101]:
filename_validation_path = INTERIM_DATA_DIR / "filename_validation.csv"

print("Exists:", filename_validation_path.exists())
print("Path:", filename_validation_path)

Exists: False
Path: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/interim/filename_validation.csv


In [102]:
for file_name in [
    "filename_validation.csv",
    "recordings.csv",
    "audio_info.csv",
    "cycles_validation.csv"
]:
    path = INTERIM_DATA_DIR / file_name
    print(
        f"{file_name}: "
        f"{'EXISTS' if path.exists() else 'MISSING'}"
    )

filename_validation.csv: MISSING
recordings.csv: EXISTS
audio_info.csv: EXISTS
cycles_validation.csv: EXISTS


In [103]:
validation_df.to_csv(
    INTERIM_DATA_DIR / "filename_validation.csv",
    index=False
)

print("Saved:", INTERIM_DATA_DIR / "filename_validation.csv")
print("Rows:", len(validation_df))

Saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/interim/filename_validation.csv
Rows: 920


In [104]:
filename_validation_path = (
    INTERIM_DATA_DIR / "filename_validation.csv"
)

print("Exists:", filename_validation_path.exists())
print("Path:", filename_validation_path)

Exists: True
Path: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/interim/filename_validation.csv


In [105]:
cycle_validation_df["wav_file"] = (
    cycle_validation_df["file"]
    .str.replace(".txt", ".wav", regex=False)
)

In [106]:
cycle_validation_df[
    ["file", "wav_file"]
].head()

,file,wav_file
0,101_1b1_Al_sc_Meditron.txt,101_1b1_Al_sc_Meditron.wav
1,101_1b1_Al_sc_Meditron.txt,101_1b1_Al_sc_Meditron.wav
2,101_1b1_Al_sc_Meditron.txt,101_1b1_Al_sc_Meditron.wav
3,101_1b1_Al_sc_Meditron.txt,101_1b1_Al_sc_Meditron.wav
4,101_1b1_Al_sc_Meditron.txt,101_1b1_Al_sc_Meditron.wav


In [107]:
cycle_with_audio = cycle_validation_df.merge(
    audio_info_df[["file", "duration"]],
    left_on="wav_file",
    right_on="file",
    how="left",
    suffixes=("", "_audio")
)

In [108]:
print(
    "Cycles without audio duration:",
    cycle_with_audio["duration"].isna().sum()
)

Cycles without audio duration: 0


In [109]:
cycle_with_audio["time_valid"] = (
    (cycle_with_audio["start_time"] >= 0) &
    (cycle_with_audio["end_time"] > cycle_with_audio["start_time"]) &
    (cycle_with_audio["end_time"] <= cycle_with_audio["duration"])
)

In [110]:
print(
    "Valid cycle times:",
    cycle_with_audio["time_valid"].sum()
)

print(
    "Invalid cycle times:",
    (~cycle_with_audio["time_valid"]).sum()
)

Valid cycle times: 6898
Invalid cycle times: 0


In [111]:
cycle_with_audio[
    cycle_with_audio["time_valid"] == False
]

,file,cycle_number,start_time,end_time,crackles,wheezes,valid,errors,wav_file,file_audio,duration,time_valid


In [113]:
cycles_df = cycle_with_audio.merge(
    recordings_df[
        [
            "patient_id",
            "recording_id",
            "chest_location",
            "acquisition_mode",
            "equipment",
            "wav_path"
        ]
    ],
    left_on="wav_file",
    right_on=recordings_df["wav_path"].map(lambda x: Path(x).name),
    how="left"
)

In [114]:
print("Total cycles:", len(cycles_df))

Total cycles: 6898


In [115]:
def get_sound_label(row):
    if row["crackles"] == 0 and row["wheezes"] == 0:
        return "Normal"
    elif row["crackles"] == 1 and row["wheezes"] == 0:
        return "Crackles"
    elif row["crackles"] == 0 and row["wheezes"] == 1:
        return "Wheezes"
    elif row["crackles"] == 1 and row["wheezes"] == 1:
        return "Both"
    else:
        return "Unknown"


cycles_df["sound_label"] = cycles_df.apply(
    get_sound_label,
    axis=1
)

In [116]:
print(cycles_df["sound_label"].value_counts())

sound_label
Normal      3642
Crackles    1864
Wheezes      886
Both         506
Name: count, dtype: int64


In [117]:
print(
    "Unknown labels:",
    (cycles_df["sound_label"] == "Unknown").sum()
)

Unknown labels: 0


In [118]:
print(
    cycles_df[
        ["crackles", "wheezes", "sound_label"]
    ].head(10)
)

   crackles  wheezes sound_label
0         0        0      Normal
1         0        0      Normal
2         0        0      Normal
3         0        0      Normal
4         0        0      Normal
5         0        0      Normal
6         0        0      Normal
7         0        0      Normal
8         0        0      Normal
9         0        0      Normal


In [119]:
print(
    cycles_df[
        [
            "patient_id",
            "recording_id",
            "chest_location",
            "acquisition_mode",
            "equipment",
            "wav_path"
        ]
    ].isnull().sum()
)

patient_id          0
recording_id        0
chest_location      0
acquisition_mode    0
equipment           0
wav_path            0
dtype: int64


In [120]:
print("Time-invalid cycles:", (~cycles_df["time_valid"]).sum())

Time-invalid cycles: 0


In [121]:
print(cycles_df.columns.tolist())

['file', 'cycle_number', 'start_time', 'end_time', 'crackles', 'wheezes', 'valid', 'errors', 'wav_file', 'file_audio', 'duration', 'time_valid', 'patient_id', 'recording_id', 'chest_location', 'acquisition_mode', 'equipment', 'wav_path', 'sound_label']


In [122]:
cycles_df.head()

,file,cycle_number,start_time,end_time,crackles,wheezes,valid,errors,wav_file,file_audio,duration,time_valid,patient_id,recording_id,chest_location,acquisition_mode,equipment,wav_path,sound_label
0,101_1b1_Al_sc_Meditron.txt,1,0.036,0.579,0,0,True,[],101_1b1_Al_sc_Meditron.wav,101_1b1_Al_sc_Meditron.wav,20.0,True,101,1b1,Al,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,Normal
1,101_1b1_Al_sc_Meditron.txt,2,0.579,2.450,0,0,True,[],101_1b1_Al_sc_Meditron.wav,101_1b1_Al_sc_Meditron.wav,20.0,True,101,1b1,Al,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,Normal
2,101_1b1_Al_sc_Meditron.txt,3,2.450,3.893,0,0,True,[],101_1b1_Al_sc_Meditron.wav,101_1b1_Al_sc_Meditron.wav,20.0,True,101,1b1,Al,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,Normal
3,101_1b1_Al_sc_Meditron.txt,4,3.893,5.793,0,0,True,[],101_1b1_Al_sc_Meditron.wav,101_1b1_Al_sc_Meditron.wav,20.0,True,101,1b1,Al,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,Normal
4,101_1b1_Al_sc_Meditron.txt,5,5.793,7.521,0,0,True,[],101_1b1_Al_sc_Meditron.wav,101_1b1_Al_sc_Meditron.wav,20.0,True,101,1b1,Al,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,Normal


In [123]:
final_cycle_columns = [
    "patient_id",
    "recording_id",
    "chest_location",
    "acquisition_mode",
    "equipment",
    "wav_path",
    "cycle_number",
    "start_time",
    "end_time",
    "crackles",
    "wheezes",
    "sound_label"
]

cycles_final_df = cycles_df[final_cycle_columns].copy()

In [124]:
print("Rows:", len(cycles_final_df))
print("Columns:", len(cycles_final_df.columns))

Rows: 6898
Columns: 12


In [125]:
cycles_final_df.head()

,patient_id,recording_id,chest_location,acquisition_mode,equipment,wav_path,cycle_number,start_time,end_time,crackles,wheezes,sound_label
0,101,1b1,Al,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,1,0.036,0.579,0,0,Normal
1,101,1b1,Al,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,2,0.579,2.450,0,0,Normal
2,101,1b1,Al,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,3,2.450,3.893,0,0,Normal
3,101,1b1,Al,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,4,3.893,5.793,0,0,Normal
4,101,1b1,Al,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,5,5.793,7.521,0,0,Normal


In [126]:
cycles_final_df.isnull().sum()

patient_id          0
recording_id        0
chest_location      0
acquisition_mode    0
equipment           0
wav_path            0
cycle_number        0
start_time          0
end_time            0
crackles            0
wheezes             0
sound_label         0
dtype: int64

In [127]:
cycles_final_df.to_csv(
    INTERIM_DATA_DIR / "cycles.csv",
    index=False
)

print("Saved:", INTERIM_DATA_DIR / "cycles.csv")

Saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/interim/cycles.csv


In [128]:
for file in sorted(RAW_DATA_DIR.iterdir()):
    print(file.name)

101_1b1_Al_sc_Meditron.txt
101_1b1_Al_sc_Meditron.wav
101_1b1_Pr_sc_Meditron.txt
101_1b1_Pr_sc_Meditron.wav
102_1b1_Ar_sc_Meditron.txt
102_1b1_Ar_sc_Meditron.wav
103_2b2_Ar_mc_LittC2SE.txt
103_2b2_Ar_mc_LittC2SE.wav
104_1b1_Al_sc_Litt3200.txt
104_1b1_Al_sc_Litt3200.wav
104_1b1_Ar_sc_Litt3200.txt
104_1b1_Ar_sc_Litt3200.wav
104_1b1_Ll_sc_Litt3200.txt
104_1b1_Ll_sc_Litt3200.wav
104_1b1_Lr_sc_Litt3200.txt
104_1b1_Lr_sc_Litt3200.wav
104_1b1_Pl_sc_Litt3200.txt
104_1b1_Pl_sc_Litt3200.wav
104_1b1_Pr_sc_Litt3200.txt
104_1b1_Pr_sc_Litt3200.wav
105_1b1_Tc_sc_Meditron.txt
105_1b1_Tc_sc_Meditron.wav
106_2b1_Pl_mc_LittC2SE.txt
106_2b1_Pl_mc_LittC2SE.wav
106_2b1_Pr_mc_LittC2SE.txt
106_2b1_Pr_mc_LittC2SE.wav
107_2b3_Al_mc_AKGC417L.txt
107_2b3_Al_mc_AKGC417L.wav
107_2b3_Ar_mc_AKGC417L.txt
107_2b3_Ar_mc_AKGC417L.wav
107_2b3_Ll_mc_AKGC417L.txt
107_2b3_Ll_mc_AKGC417L.wav
107_2b3_Lr_mc_AKGC417L.txt
107_2b3_Lr_mc_AKGC417L.wav
107_2b3_Pl_mc_AKGC417L.txt
107_2b3_Pl_mc_AKGC417L.wav
107_2b3_Pr_mc_AKGC417L.txt
1

In [129]:
other_files = [
    file for file in RAW_DATA_DIR.iterdir()
    if file.is_file() and file.suffix.lower() not in {".wav", ".txt", ".sh"}
]

print("Other files:", len(other_files))

for file in sorted(other_files):
    print(file.name)

Other files: 0


In [132]:
METADATA_DIR = RAW_DATA_DIR / "metadata"

METADATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Metadata directory:", METADATA_DIR)


Metadata directory: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/raw/ICBHI/metadata


In [134]:
diagnosis_df = pd.read_csv(
    METADATA_DIR / "diagnosis.csv"
)

print("Total diagnosis records:", len(diagnosis_df))
print("Unique patients:", diagnosis_df["patient_id"].nunique())

Total diagnosis records: 126
Unique patients: 126


In [135]:
print(diagnosis_df.head())

   patient_id diagnosis
0         101      URTI
1         102   Healthy
2         103    Asthma
3         104      COPD
4         105      URTI


In [136]:
print(diagnosis_df["diagnosis"].value_counts())

diagnosis
COPD              64
Healthy           26
URTI              14
Bronchiectasis     7
Pneumonia          6
Bronchiolitis      6
LRTI               2
Asthma             1
Name: count, dtype: int64


In [137]:
print("\nMissing values:")
print(diagnosis_df.isnull().sum())


Missing values:
patient_id    0
diagnosis     0
dtype: int64


In [138]:
missing_diagnosis_patients = set(
    recordings_df["patient_id"]
) - set(
    diagnosis_df["patient_id"]
)

print(
    "Patients without diagnosis:",
    sorted(missing_diagnosis_patients)
)

Patients without diagnosis: []


In [139]:
patients_df = diagnosis_df.copy()

print("Patients:", len(patients_df))
print("Unique patient IDs:", patients_df["patient_id"].nunique())

Patients: 126
Unique patient IDs: 126


In [140]:
print(patients_df["diagnosis"].value_counts())

diagnosis
COPD              64
Healthy           26
URTI              14
Bronchiectasis     7
Pneumonia          6
Bronchiolitis      6
LRTI               2
Asthma             1
Name: count, dtype: int64


In [141]:
print(
    "Minimum patient ID:",
    patients_df["patient_id"].min()
)

print(
    "Maximum patient ID:",
    patients_df["patient_id"].max()
)

Minimum patient ID: 101
Maximum patient ID: 226


In [142]:
patients_df.to_csv(
    INTERIM_DATA_DIR / "patients.csv",
    index=False
)

print("Saved:", INTERIM_DATA_DIR / "patients.csv")

Saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/interim/patients.csv


In [143]:
patients_df = diagnosis_df.copy()

print("Patients:", len(patients_df))
print("Unique patient IDs:", patients_df["patient_id"].nunique())

Patients: 126
Unique patient IDs: 126


In [144]:
print(patients_df["diagnosis"].value_counts())

diagnosis
COPD              64
Healthy           26
URTI              14
Bronchiectasis     7
Pneumonia          6
Bronchiolitis      6
LRTI               2
Asthma             1
Name: count, dtype: int64


In [145]:
print("Minimum patient ID:", patients_df["patient_id"].min())
print("Maximum patient ID:", patients_df["patient_id"].max())

Minimum patient ID: 101
Maximum patient ID: 226


In [146]:
patients_df.to_csv(
    INTERIM_DATA_DIR / "patients.csv",
    index=False
)

print("Saved:", INTERIM_DATA_DIR / "patients.csv")

Saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/interim/patients.csv


In [147]:
demographic_path = RAW_DATA_DIR / "metadata" / "demographic.csv"

demographic_df = pd.read_csv(demographic_path)

print("Rows:", len(demographic_df))
print("Unique patient IDs:", demographic_df["patient_id"].nunique())
print("Minimum patient ID:", demographic_df["patient_id"].min())
print("Maximum patient ID:", demographic_df["patient_id"].max())

print("\nColumns:")
print(demographic_df.columns.tolist())

display(demographic_df.head())

Rows: 126
Unique patient IDs: 126
Minimum patient ID: 101
Maximum patient ID: 226

Columns:
['patient_id', 'diagnosis', 'age', 'sex', 'adult_bmi', 'child_weight', 'child_height']


,patient_id,diagnosis,age,sex,adult_bmi,child_weight,child_height
0,101,URTI,3.00,F,NaN,19.0,99.0
1,102,Healthy,0.75,F,NaN,9.8,73.0
2,103,Asthma,70.00,F,33.00,NaN,NaN
3,104,COPD,70.00,F,28.47,NaN,NaN
4,105,URTI,7.00,F,NaN,32.0,135.0


In [148]:
print(demographic_df.isnull().sum())

patient_id       0
diagnosis        0
age              1
sex              1
adult_bmi       51
child_weight    82
child_height    84
dtype: int64


In [149]:
duplicate_patients = demographic_df[
    demographic_df["patient_id"].duplicated(keep=False)
]

print("Duplicate patient rows:", len(duplicate_patients))

if len(duplicate_patients) > 0:
    display(duplicate_patients)

Duplicate patient rows: 0


In [150]:
diagnosis_check = patients_df.merge(
    demographic_df[["patient_id", "diagnosis"]],
    on="patient_id",
    how="outer",
    suffixes=("_original", "_demographic"),
    indicator=True
)

print(diagnosis_check["_merge"].value_counts())

mismatches = diagnosis_check[
    diagnosis_check["diagnosis_original"] != diagnosis_check["diagnosis_demographic"]
]

print("Diagnosis mismatches:", len(mismatches))

if len(mismatches) > 0:
    display(mismatches)

_merge
both          126
left_only       0
right_only      0
Name: count, dtype: int64
Diagnosis mismatches: 0


In [151]:
patients_final_df = demographic_df.copy()

print("Final patient records:", len(patients_final_df))
print("Unique patients:", patients_final_df["patient_id"].nunique())

display(patients_final_df.head())

Final patient records: 126
Unique patients: 126


,patient_id,diagnosis,age,sex,adult_bmi,child_weight,child_height
0,101,URTI,3.00,F,NaN,19.0,99.0
1,102,Healthy,0.75,F,NaN,9.8,73.0
2,103,Asthma,70.00,F,33.00,NaN,NaN
3,104,COPD,70.00,F,28.47,NaN,NaN
4,105,URTI,7.00,F,NaN,32.0,135.0


In [152]:
patients_final_df.to_csv(
    INTERIM_DATA_DIR / "patients.csv",
    index=False
)

print("Saved:", INTERIM_DATA_DIR / "patients.csv")

Saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/interim/patients.csv


In [153]:
print("Missing values in patient metadata:")
print(patients_final_df.isnull().sum())

Missing values in patient metadata:
patient_id       0
diagnosis        0
age              1
sex              1
adult_bmi       51
child_weight    82
child_height    84
dtype: int64


In [154]:
print("\nData types:")
print(patients_final_df.dtypes)


Data types:
patient_id        int64
diagnosis           str
age             float64
sex                 str
adult_bmi       float64
child_weight    float64
child_height    float64
dtype: object


In [155]:
print("\nSex distribution:")
print(patients_final_df["sex"].value_counts())

print("\nDiagnosis distribution:")
print(patients_final_df["diagnosis"].value_counts())


Sex distribution:
sex
M    79
F    46
Name: count, dtype: int64

Diagnosis distribution:
diagnosis
COPD              64
Healthy           26
URTI              14
Bronchiectasis     7
Pneumonia          6
Bronchiolitis      6
LRTI               2
Asthma             1
Name: count, dtype: int64


In [156]:
missing_basic = patients_final_df[
    patients_final_df["age"].isna() |
    patients_final_df["sex"].isna()
]

display(missing_basic)

,patient_id,diagnosis,age,sex,adult_bmi,child_weight,child_height
122,223,COPD,NaN,NaN,NaN,NaN,NaN


In [157]:
patient_223_recordings = recordings_df[
    recordings_df["patient_id"] == 223
]

print("Recordings for patient 223:", len(patient_223_recordings))
display(patient_223_recordings)

Recordings for patient 223: 6


,patient_id,recording_id,chest_location,acquisition_mode,equipment,wav_path
908,223,1b1,Al,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
909,223,1b1,Ar,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
910,223,1b1,Ll,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
911,223,1b1,Lr,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
912,223,1b1,Pl,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...
913,223,1b1,Pr,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...


In [158]:
patient_223_cycles = cycles_final_df[
    cycles_final_df["patient_id"] == 223
]

print("Cycles for patient 223:", len(patient_223_cycles))
display(patient_223_cycles.head())

Cycles for patient 223: 61


,patient_id,recording_id,chest_location,acquisition_mode,equipment,wav_path,cycle_number,start_time,end_time,crackles,wheezes,sound_label
6778,223,1b1,Al,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,1,1.2576,3.7071,0,0,Normal
6779,223,1b1,Al,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,2,3.7071,6.5777,0,0,Normal
6780,223,1b1,Al,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,3,6.5777,9.6506,0,0,Normal
6781,223,1b1,Al,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,4,9.6506,12.8160,0,0,Normal
6782,223,1b1,Al,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,5,12.8160,15.9990,0,0,Normal


In [159]:
official_split_path = RAW_DATA_DIR / "metadata" / "official_split.csv"

official_split_df = pd.read_csv(
    official_split_path,
    sep=None,
    engine="python"
)

print("Rows:", len(official_split_df))
print("Columns:", official_split_df.columns.tolist())

display(official_split_df.head())

Rows: 920
Columns: ['recording_id', 'split']


,recording_id,split
0,101_1b1_Al_sc_Meditron,test
1,101_1b1_Pr_sc_Meditron,test
2,102_1b1_Ar_sc_Meditron,test
3,103_2b2_Ar_mc_LittC2SE,train
4,104_1b1_Al_sc_Litt3200,test


In [160]:
print(official_split_df.iloc[:, -1].value_counts())

split
train    539
test     381
Name: count, dtype: int64


In [161]:
print("Unique recordings:", official_split_df.iloc[:, 0].nunique())
print("Duplicate rows:", official_split_df.duplicated().sum())

Unique recordings: 920
Duplicate rows: 0


In [162]:
official_split_df = official_split_df.copy()

official_split_df["filename"] = (
    official_split_df.iloc[:, 0].astype(str) + ".wav"
)

print("Official split recordings:", len(official_split_df))
display(official_split_df.head())

Official split recordings: 920


,recording_id,split,filename
0,101_1b1_Al_sc_Meditron,test,101_1b1_Al_sc_Meditron.wav
1,101_1b1_Pr_sc_Meditron,test,101_1b1_Pr_sc_Meditron.wav
2,102_1b1_Ar_sc_Meditron,test,102_1b1_Ar_sc_Meditron.wav
3,103_2b2_Ar_mc_LittC2SE,train,103_2b2_Ar_mc_LittC2SE.wav
4,104_1b1_Al_sc_Litt3200,test,104_1b1_Al_sc_Litt3200.wav


In [163]:
our_recordings = set(
    Path(path).name
    for path in recordings_df["wav_path"]
)

official_recordings = set(
    official_split_df["filename"]
)

missing_from_split = our_recordings - official_recordings
missing_from_dataset = official_recordings - our_recordings

print("Our WAV recordings:", len(our_recordings))
print("Official split recordings:", len(official_recordings))
print("WAVs missing from official split:", len(missing_from_split))
print("Official recordings missing from WAV dataset:", len(missing_from_dataset))

Our WAV recordings: 920
Official split recordings: 920
WAVs missing from official split: 1
Official recordings missing from WAV dataset: 1


In [164]:
print("WAV file missing from official split:")
print(missing_from_split)

print("\nOfficial split entry missing from WAV dataset:")
print(missing_from_dataset)

WAV file missing from official split:
{'226_1b1_Pl_sc_LittC2SE.wav'}

Official split entry missing from WAV dataset:
{'226_1b1_Pl_sc_Meditron.wav'}


In [165]:
print("\nDetailed comparison:")

for x in missing_from_split:
    print("Dataset only:", repr(x))

for x in missing_from_dataset:
    print("Split only:", repr(x))


Detailed comparison:
Dataset only: '226_1b1_Pl_sc_LittC2SE.wav'
Split only: '226_1b1_Pl_sc_Meditron.wav'


In [166]:
patient_226_files = [
    p.name
    for p in (RAW_DATA_DIR).glob("226_1b1_Pl_sc_*")
]

for filename in patient_226_files:
    print(filename)

226_1b1_Pl_sc_LittC2SE.wav
226_1b1_Pl_sc_LittC2SE.txt


In [167]:
print(
    "Actual WAV exists:",
    (RAW_DATA_DIR / "226_1b1_Pl_sc_LittC2SE.wav").exists()
)

print(
    "Meditron WAV exists:",
    (RAW_DATA_DIR / "226_1b1_Pl_sc_Meditron.wav").exists()
)

Actual WAV exists: True
Meditron WAV exists: False


In [168]:
official_split_df.loc[
    official_split_df.iloc[:, 0] == "226_1b1_Pl_sc_Meditron",
    official_split_df.columns[0]
] = "226_1b1_Pl_sc_LittC2SE"

print(
    official_split_df[
        official_split_df.iloc[:, 0].str.startswith("226_1b1_Pl_sc_")
    ]
)

               recording_id  split                    filename
919  226_1b1_Pl_sc_LittC2SE  train  226_1b1_Pl_sc_Meditron.wav


In [169]:
official_split_df["filename"] = (
    official_split_df.iloc[:, 0].astype(str) + ".wav"
)

official_recordings = set(official_split_df["filename"])

missing_from_split = our_recordings - official_recordings
missing_from_dataset = official_recordings - our_recordings

print("Our WAV recordings:", len(our_recordings))
print("Official split recordings:", len(official_recordings))
print("WAVs missing from official split:", len(missing_from_split))
print("Official recordings missing from WAV dataset:", len(missing_from_dataset))

Our WAV recordings: 920
Official split recordings: 920
WAVs missing from official split: 0
Official recordings missing from WAV dataset: 0


In [170]:
split_column = official_split_df.columns[1]

split_lookup = official_split_df[
    [official_split_df.columns[0], split_column]
].copy()

split_lookup.columns = ["recording_name", "official_split"]

recordings_df["recording_name"] = recordings_df["wav_path"].apply(
    lambda x: Path(x).name
)

recordings_df = recordings_df.merge(
    split_lookup,
    on="recording_name",
    how="left"
)

print("Recordings:", len(recordings_df))
print(
    "Missing official split labels:",
    recordings_df["official_split"].isna().sum()
)

print("\nOfficial split distribution:")
print(recordings_df["official_split"].value_counts())

Recordings: 920
Missing official split labels: 920

Official split distribution:
Series([], Name: count, dtype: int64)


In [171]:
recordings_df.to_csv(
    INTERIM_DATA_DIR / "recordings.csv",
    index=False
)

print("Saved:", INTERIM_DATA_DIR / "recordings.csv")

Saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/interim/recordings.csv


In [172]:
train_patients = set(
    recordings_df.loc[
        recordings_df["official_split"] == "train",
        "patient_id"
    ]
)

test_patients = set(
    recordings_df.loc[
        recordings_df["official_split"] == "test",
        "patient_id"
    ]
)

overlap_patients = train_patients & test_patients

print("Train patients:", len(train_patients))
print("Test patients:", len(test_patients))
print("Patients appearing in both:", len(overlap_patients))
print("Overlapping patient IDs:", sorted(overlap_patients))

Train patients: 0
Test patients: 0
Patients appearing in both: 0
Overlapping patient IDs: []


In [173]:
recording_columns = [
    "patient_id",
    "recording_id",
    "chest_location",
    "acquisition_mode",
    "equipment",
    "wav_path",
    "official_split"
]

recordings_final_df = recordings_df[recording_columns].copy()

print("Rows:", len(recordings_final_df))
print("Unique recordings:", recordings_final_df["wav_path"].nunique())
print("Missing values:")
print(recordings_final_df.isnull().sum())

Rows: 920
Unique recordings: 920
Missing values:
patient_id            0
recording_id          0
chest_location        0
acquisition_mode      0
equipment             0
wav_path              0
official_split      920
dtype: int64


In [174]:
recordings_final_df.to_csv(
    INTERIM_DATA_DIR / "recordings.csv",
    index=False
)

print("Saved:", INTERIM_DATA_DIR / "recordings.csv")

Saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/interim/recordings.csv


In [175]:
missing_split = recordings_df[
    recordings_df["official_split"].isna()
]

print("Recordings with missing split:",
      len(missing_split))

display(missing_split)

Recordings with missing split: 920


,patient_id,recording_id,chest_location,acquisition_mode,equipment,wav_path,recording_name,official_split
0,101,1b1,Al,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,101_1b1_Al_sc_Meditron.wav,NaN
1,101,1b1,Pr,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,101_1b1_Pr_sc_Meditron.wav,NaN
2,102,1b1,Ar,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,102_1b1_Ar_sc_Meditron.wav,NaN
3,103,2b2,Ar,mc,LittC2SE,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,103_2b2_Ar_mc_LittC2SE.wav,NaN
4,104,1b1,Al,sc,Litt3200,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,104_1b1_Al_sc_Litt3200.wav,NaN
...,...,...,...,...,...,...,...,...
915,224,1b2,Al,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,224_1b2_Al_sc_Meditron.wav,NaN
916,225,1b1,Pl,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,225_1b1_Pl_sc_Meditron.wav,NaN
917,226,1b1,Al,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,226_1b1_Al_sc_Meditron.wav,NaN
918,226,1b1,Ll,sc,Meditron,/Users/vs/Sem 5/Respiratory-Disease-Classifica...,226_1b1_Ll_sc_Meditron.wav,NaN


In [176]:
print("Official split columns:")
print(official_split_df.columns.tolist())

print("\nFirst 5 rows:")
display(official_split_df.head())

Official split columns:
['recording_id', 'split', 'filename']

First 5 rows:


,recording_id,split,filename
0,101_1b1_Al_sc_Meditron,test,101_1b1_Al_sc_Meditron.wav
1,101_1b1_Pr_sc_Meditron,test,101_1b1_Pr_sc_Meditron.wav
2,102_1b1_Ar_sc_Meditron,test,102_1b1_Ar_sc_Meditron.wav
3,103_2b2_Ar_mc_LittC2SE,train,103_2b2_Ar_mc_LittC2SE.wav
4,104_1b1_Al_sc_Litt3200,test,104_1b1_Al_sc_Litt3200.wav


In [177]:
print("Recordings columns:")
print(recordings_df.columns.tolist())

print("\nFirst 5 recording names:")
display(
    recordings_df[
        ["recording_name", "official_split"]
    ].head()
)

Recordings columns:
['patient_id', 'recording_id', 'chest_location', 'acquisition_mode', 'equipment', 'wav_path', 'recording_name', 'official_split']

First 5 recording names:


,recording_name,official_split
0,101_1b1_Al_sc_Meditron.wav,NaN
1,101_1b1_Pr_sc_Meditron.wav,NaN
2,102_1b1_Ar_sc_Meditron.wav,NaN
3,103_2b2_Ar_mc_LittC2SE.wav,NaN
4,104_1b1_Al_sc_Litt3200.wav,NaN


In [178]:
recordings_df = recordings_df.drop(
    columns=["official_split"],
    errors="ignore"
)

In [179]:
split_lookup = official_split_df[
    ["filename", "split"]
].copy()

split_lookup = split_lookup.rename(
    columns={
        "filename": "recording_name",
        "split": "official_split"
    }
)

print(split_lookup.head())

               recording_name official_split
0  101_1b1_Al_sc_Meditron.wav           test
1  101_1b1_Pr_sc_Meditron.wav           test
2  102_1b1_Ar_sc_Meditron.wav           test
3  103_2b2_Ar_mc_LittC2SE.wav          train
4  104_1b1_Al_sc_Litt3200.wav           test


In [180]:
recordings_df = recordings_df.merge(
    split_lookup,
    on="recording_name",
    how="left",
    validate="one_to_one"
)

print("Recordings:", len(recordings_df))
print(
    "Missing official split labels:",
    recordings_df["official_split"].isna().sum()
)

print("\nSplit distribution:")
print(recordings_df["official_split"].value_counts())

Recordings: 920
Missing official split labels: 0

Split distribution:
official_split
train    539
test     381
Name: count, dtype: int64


In [181]:
recording_columns = [
    "patient_id",
    "recording_id",
    "chest_location",
    "acquisition_mode",
    "equipment",
    "wav_path",
    "official_split"
]

recordings_final_df = recordings_df[recording_columns].copy()

print("Rows:", len(recordings_final_df))
print("Unique recordings:", recordings_final_df["wav_path"].nunique())

print("\nMissing values:")
print(recordings_final_df.isnull().sum())

Rows: 920
Unique recordings: 920

Missing values:
patient_id          0
recording_id        0
chest_location      0
acquisition_mode    0
equipment           0
wav_path            0
official_split      0
dtype: int64


In [182]:
recordings_final_df.to_csv(
    INTERIM_DATA_DIR / "recordings.csv",
    index=False
)

print("Saved:", INTERIM_DATA_DIR / "recordings.csv")

Saved: /Users/vs/Sem 5/Respiratory-Disease-Classification/data/interim/recordings.csv


In [183]:
print(
    recordings_final_df["official_split"].value_counts()
)

official_split
train    539
test     381
Name: count, dtype: int64


In [184]:
train_patients = set(
    recordings_final_df.loc[
        recordings_final_df["official_split"] == "train",
        "patient_id"
    ]
)

test_patients = set(
    recordings_final_df.loc[
        recordings_final_df["official_split"] == "test",
        "patient_id"
    ]
)

overlap_patients = train_patients & test_patients

print("Train patients:", len(train_patients))
print("Test patients:", len(test_patients))
print("Patients in both train and test:", len(overlap_patients))
print("Overlapping patients:", sorted(overlap_patients))

Train patients: 79
Test patients: 49
Patients in both train and test: 2
Overlapping patients: [156, 218]


In [185]:
print("=== FINAL RELATIONSHIP CHECK ===")

print("\nPatients:")
print("Rows:", len(patients_final_df))
print("Unique patient IDs:",
      patients_final_df["patient_id"].nunique())

print("\nRecordings:")
print("Rows:", len(recordings_final_df))
print("Unique recording paths:",
      recordings_final_df["wav_path"].nunique())

print("\nCycles:")
print("Rows:", len(cycles_final_df))

print("\nPatients in recordings but not patient table:")
print(
    set(recordings_final_df["patient_id"])
    - set(patients_final_df["patient_id"])
)

print("\nPatients in cycles but not patient table:")
print(
    set(cycles_final_df["patient_id"])
    - set(patients_final_df["patient_id"])
)

print("\nRecordings in cycles but not recordings table:")
cycle_recordings = set(
    zip(
        cycles_final_df["patient_id"],
        cycles_final_df["recording_id"]
    )
)

recording_keys = set(
    zip(
        recordings_final_df["patient_id"],
        recordings_final_df["recording_id"]
    )
)

print(len(cycle_recordings - recording_keys))

=== FINAL RELATIONSHIP CHECK ===

Patients:
Rows: 126
Unique patient IDs: 126

Recordings:
Rows: 920
Unique recording paths: 920

Cycles:
Rows: 6898

Patients in recordings but not patient table:
set()

Patients in cycles but not patient table:
set()

Recordings in cycles but not recordings table:
0


In [186]:
dataset_summary = {
    "patients": patients_final_df["patient_id"].nunique(),
    "recordings": recordings_final_df["wav_path"].nunique(),
    "respiratory_cycles": len(cycles_final_df),
    "train_recordings": (
        recordings_final_df["official_split"] == "train"
    ).sum(),
    "test_recordings": (
        recordings_final_df["official_split"] == "test"
    ).sum(),
    "train_patients": len(train_patients),
    "test_patients": len(test_patients),
    "overlapping_patients": len(overlap_patients),
}

dataset_summary

{'patients': 126,
 'recordings': 920,
 'respiratory_cycles': 6898,
 'train_recordings': np.int64(539),
 'test_recordings': np.int64(381),
 'train_patients': 79,
 'test_patients': 49,
 'overlapping_patients': 2}

In [ ]:
## Dataset Understanding — Final Summary

The ICBHI respiratory sound dataset used in this project contains:

- 126 patients
- 920 respiratory sound recordings
- 6,898 annotated respiratory cycles

### Patient-level information

Each patient has a diagnosis and available demographic information.
Diagnosis is a patient-level target and is kept separate from
respiratory-cycle sound labels.

### Recording-level information

Each recording contains:

- patient ID
- recording ID
- chest location
- acquisition mode
- recording equipment
- audio path
- official ICBHI train/test assignment

The official recording-level split contains 79 patients with training
recordings and 49 patients with test recordings. Patients 156 and 218
have recordings in both partitions.

### Cycle-level information

Each respiratory cycle contains:

- start time
- end time
- crackle annotation
- wheeze annotation
- derived sound label

The sound labels are:

- Normal
- Crackles
- Wheezes
- Both

### Important modeling decision

Respiratory sound labels and disease diagnoses are treated as two
different prediction problems.

The official ICBHI recording-level split will be retained for
respiratory sound classification benchmarking.

Disease classification will use a patient-level split so that recordings
from the same patient cannot appear in both training and testing.

### Data integrity

All 126 patients are represented in the recording/cycle data, all
recordings are unique, and all 6,898 cycles reference valid recordings.
No orphan patient or recording relationships were found.

Raw WAV and annotation files remain unchanged.